<a href="https://colab.research.google.com/github/Mastermind0309/NTUH_HCC/blob/main/FixPixel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive (skip if already mounted)
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pillow

import os, math
from PIL import Image

# ====== CONSTANTS ======
MAX_PIXELS = 40_000_000
TARGET_DPI = 1200

# ====== PATHS ======
INPUT_FILE = "/content/drive/MyDrive/HCC/TIF/Figure2_1_1200dpi.tif"   # your specific file
OUTPUT_DIR = "/content/drive/MyDrive/HCC/TIF_out"           # output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== FUNCTIONS ======
def check_tiff(path, max_pixels=MAX_PIXELS, target_dpi=TARGET_DPI):
    with Image.open(path) as im:
        w, h = im.size
        dpi = im.info.get("dpi", (None, None))
        total = w * h
        print(f"\n📂 File: {os.path.basename(path)}")
        print(f" - Dimensions : {w} × {h} px  ({total/1e6:.2f} MP)")
        print(f" - Mode       : {im.mode}")
        print(f" - DPI        : {dpi}")
        print(f" - Compression: {im.info.get('compression','none')}")
        print(" - Pixel limit:", " OK" if total <= max_pixels else f" {total/1e6:.2f} MP > 40.00 MP")
        print(" - DPI check  :", " 1200x1200" if dpi==(target_dpi,target_dpi) else f" {dpi}")

def safe_size_under_limit(w, h, limit=MAX_PIXELS):
    total = w * h
    if total <= limit:
        return w, h, 1.0
    scale = math.sqrt(limit / float(total))
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    while new_w * new_h > limit:
        if new_w >= new_h:
            new_w -= 1
            new_h = max(1, int(round(new_w * (h / w))))
        else:
            new_h -= 1
            new_w = max(1, int(round(new_h * (w / h))))
    return new_w, new_h, scale

def resize_tiff(in_path, out_dir=OUTPUT_DIR, target_dpi=TARGET_DPI, limit=MAX_PIXELS):
    base = os.path.splitext(os.path.basename(in_path))[0]
    out_path = os.path.join(out_dir, f"{base}_1200dpi_resized.tif")

    with Image.open(in_path) as im:
        w, h = im.size
        new_w, new_h, scale = safe_size_under_limit(w, h, limit)
        resample = Image.NEAREST if im.mode == "1" else Image.LANCZOS
        if (new_w, new_h) != (w, h):
            im = im.resize((new_w, new_h), resample=resample)
        im.save(out_path, format="TIFF", dpi=(target_dpi, target_dpi), compression="tiff_lzw")  # ✅ fixed line

    return out_path

# ====== RUN ======
print("\n--- BEFORE ---")
check_tiff(INPUT_FILE)

output_path = resize_tiff(INPUT_FILE)
print("\n--- AFTER ---")
check_tiff(output_path)

print("\n Saved resized file to:", output_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- BEFORE ---

📂 File: Figure2_1_1200dpi.tif
 - Dimensions : 8526 × 9452 px  (80.59 MP)
 - Mode       : RGB
 - DPI        : (1200.0, 1200.0)
 - Compression: tiff_lzw
 - Pixel limit:  80.59 MP > 40.00 MP
 - DPI check  :  1200x1200

--- AFTER ---

📂 File: Figure2_1_1200dpi_1200dpi_resized.tif
 - Dimensions : 6006 × 6658 px  (39.99 MP)
 - Mode       : RGB
 - DPI        : (1200.0, 1200.0)
 - Compression: tiff_lzw
 - Pixel limit:  OK
 - DPI check  :  1200x1200

 Saved resized file to: /content/drive/MyDrive/HCC/TIF_out/Figure2_1_1200dpi_1200dpi_resized.tif
